# ตอนที่ 0: แม่แบบของซีรีส์ (Template)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kobkrit/thai-llm-tutorials/blob/main/notebooks/_template.ipynb)

โน้ตบุ๊กแม่แบบ: คัดลอกไฟล์นี้เพื่อเริ่มเขียนตอนใหม่ โครงสร้าง Cell 0 และ Cell 1 ห้ามแก้

โดย **ดร.กอบกฤตย์ วิริยะยุทธกร** — ผู้สร้าง OpenThaiGPT, CEO บริษัท iApp Technology
บทความฉบับเต็ม: [kobkrit.com](https://kobkrit.com)

---

## โครงของบทความตอนนี้

1. ปัญหาคืออะไร และทำไมต้องแก้
2. ตั้งค่า และวัด baseline ก่อนลงมือทำอะไรทั้งสิ้น
3. ทฤษฎีเท่าที่จำเป็น
4. เตรียมข้อมูล
5. ลงมือทำ (implementation)
6. รันจริง
7. วัดผลซ้ำด้วยชุดวัดเดิม
8. ตารางผลลัพธ์
9. วิเคราะห์: อะไรได้ผล อะไรไม่ได้ผล
10. สรุป และตอนต่อไป

---

## ก่อนเริ่ม

- โน้ตบุ๊กนี้ออกแบบมาให้รันได้บน **Google Colab แบบฟรี (Tesla T4)**
- T4 เป็นสถาปัตยกรรม Turing (sm_75) ซึ่ง **ไม่รองรับ bfloat16 และไม่รองรับ FlashAttention-2**
  เราจึงใช้ `torch_dtype=torch.float16`, `attn_implementation="sdpa"` และ `fp16=True` เสมอ
  (รายละเอียดอยู่ใน Cell 0 ด้านล่าง — อ่านคอมเมนต์ให้ครบ)
- ทุกตอนในซีรีส์นี้ใช้ชุดวัดผลกลางตัวเดียวกันคือ `kobeval` และเบนช์มาร์ก **KobEval-TH**
  (120 ข้อ, 4 slice) เพื่อให้ตัวเลขของแต่ละตอนเทียบกันได้จริง


In [ ]:
# =============================================================================
# CELL 0 -- SETUP
# ชุดนี้ต้อง "เหมือนกันทุกตัวอักษร" (byte-identical) ในโน้ตบุ๊กทั้ง 10 ตอน
# ตรวจสอบด้วย: python3 scripts/check_cell0.py
# =============================================================================
# ทำไมต้องเหมือนกันทุกตอน: ถ้า setup ต่างกันแม้นิดเดียว ตัวเลขที่วัดได้จาก
# แต่ละตอนจะเปรียบเทียบกันไม่ได้ ซึ่งทำให้ทั้งซีรีส์นี้ไร้ความหมาย
# -----------------------------------------------------------------------------

# 1) ติดตั้ง dependencies (pin เวอร์ชันไว้ทั้งหมด เพื่อให้ผลลัพธ์ทำซ้ำได้)
#    หมายเหตุ: เราจงใจ "ไม่" ติดตั้ง torch ใหม่ เพราะ Colab มี torch ที่ build
#    มาให้ตรงกับ CUDA driver อยู่แล้ว การ pip install torch ทับคือสาเหตุอันดับ
#    หนึ่งที่ทำให้ notebook พังบน Colab
!pip install -q \
    "transformers==4.51.3" \
    "accelerate==1.6.0" \
    "datasets==3.5.0" \
    "peft==0.15.1" \
    "trl==0.16.1" \
    "bitsandbytes==0.45.5" \
    "sentencepiece==0.2.0" \
    "matplotlib==3.10.1"

# 2) ติดตั้งฟอนต์ไทยให้ matplotlib (ไม่งั้นกราฟจะขึ้นเป็นสี่เหลี่ยม tofu)
!apt-get install -y fonts-thai-tlwg > /dev/null 2>&1

# 3) ดึง repo (ได้ทั้งแพ็กเกจ kobeval และชุดข้อมูล KobEval-TH)
import os
if not os.path.exists("/content/thai-llm-tutorials"):
    !git clone -q https://github.com/kobkrit/thai-llm-tutorials.git /content/thai-llm-tutorials
!pip install -q -e /content/thai-llm-tutorials

import random
import numpy as np
import torch

# 4) ยืนยันว่ามี GPU จริง -- ถ้าไม่มีให้หยุดตรงนี้เลย ดีกว่าไปพังตอน train
assert torch.cuda.is_available(), (
    "ไม่พบ GPU! ไปที่ Runtime > Change runtime type > Hardware accelerator > GPU "
    "แล้วรันเซลล์นี้ใหม่"
)

GPU_NAME = torch.cuda.get_device_name(0)
CAPABILITY = torch.cuda.get_device_capability(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
SUPPORTS_BF16 = torch.cuda.is_bf16_supported()

print(f"GPU            : {GPU_NAME}")
print(f"Compute cap.   : sm_{CAPABILITY[0]}{CAPABILITY[1]}")
print(f"VRAM           : {VRAM_GB:.1f} GB")
print(f"SUPPORTS_BF16  : {SUPPORTS_BF16}")
print(f"torch          : {torch.__version__}")

# -----------------------------------------------------------------------------
# 5) จุดสำคัญที่สุดของเซลล์นี้ -- อ่านให้ครบ
#
# ถ้าคุณได้ Tesla T4 (ซึ่งเป็นค่าปกติของ Colab ฟรี) คุณจะเห็น
#     SUPPORTS_BF16 : False
#
# T4 เป็นสถาปัตยกรรม Turing (sm_75) ซึ่ง "ไม่รองรับ" สองอย่างนี้:
#   - bfloat16  -> ต้องใช้ float16 แทน
#   - FlashAttention-2 -> ต้องใช้ sdpa แทน
#
# กับดักที่คนเจอบ่อยที่สุด: config ของ Qwen3-0.6B ระบุ torch_dtype: bfloat16
# ไว้ในไฟล์ ดังนั้นถ้าเขียน torch_dtype="auto" transformers จะเชื่อ config
# แล้วโหลดเป็น bf16 บนการ์ดที่ไม่รองรับ ผลคือได้ NaN, loss ไม่ลด หรือ
# โมเดลพ่นข้อความมั่ว โดยไม่มี error ฟ้องให้เห็นเลย
#
# เราจึงกำหนดค่าพวกนี้เอง "อย่างชัดเจน" ทุกครั้ง ไม่พึ่ง auto
# -----------------------------------------------------------------------------
DTYPE = torch.bfloat16 if SUPPORTS_BF16 else torch.float16
ATTN_IMPL = "sdpa"          # ห้ามใช้ flash_attention_2 บน T4
USE_FP16 = not SUPPORTS_BF16  # ส่งเข้า TrainingArguments: fp16=USE_FP16, bf16=SUPPORTS_BF16

print(f"\n--> DTYPE      : {DTYPE}")
print(f"--> ATTN_IMPL  : {ATTN_IMPL}")
print(f"--> fp16={USE_FP16}, bf16={SUPPORTS_BF16}  (ใช้คู่นี้กับ TrainingArguments)")

# 6) ตั้ง seed ทุกตัวที่เกี่ยวข้อง เพื่อให้ผลลัพธ์ทำซ้ำได้
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# 7) import kobeval -- ชุดวัดผลกลางที่ใช้เหมือนกันทั้ง 10 ตอน
from kobeval import evaluate, compare, plot_before_after, th_ratio, wilson_ci
from kobeval import EVAL_CONTRACT, verify_checksums

print(f"\nkobeval contract: {EVAL_CONTRACT}")
print(f"KobEval-TH checksum ok: {verify_checksums()['ok']}")

---

## 2. ตั้งค่า และวัด baseline ก่อนลงมือทำอะไรทั้งสิ้น

หลักการของทั้งซีรีส์: **วัดก่อนเสมอ**

เซลล์ถัดไปคือ Cell 1 ซึ่งวัดผลโมเดลตั้งต้นด้วย KobEval-TH ก่อนที่เราจะเทรน
หรือแก้อะไรทั้งนั้น ถ้าไม่มีตัวเลขตั้งต้น เราจะไม่มีทางรู้เลยว่าสิ่งที่ทำในตอนนี้
ทำให้ดีขึ้นจริงหรือแค่รู้สึกว่าดีขึ้น

สังเกตค่า `th_ratio` เป็นพิเศษ — มันคือสัดส่วนตัวอักษรไทยในคำตอบ
จุดที่ Qwen3-0.6B พังบ่อยที่สุดกับคำถามภาษาไทย ไม่ใช่การตอบผิด
แต่คือการ **ตอบเป็นภาษาอังกฤษหรือภาษาจีนอย่างมั่นใจ** ซึ่ง accuracy อย่างเดียวมองไม่เห็น


In [ ]:
# =============================================================================
# CELL 1 -- BASELINE (วัดผลก่อนเทรน/ก่อนแก้อะไรทั้งสิ้น)
# เซลล์นี้เป็นเซลล์โค้ดที่สองของทุกตอนในซีรีส์
# =============================================================================
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,             # <-- ห้ามใช้ "auto" เด็ดขาดบน T4 (ดูคอมเมนต์ Cell 0)
    attn_implementation=ATTN_IMPL, # <-- "sdpa" ไม่ใช่ flash_attention_2
    device_map="cuda:0",
)
model.eval()

print(f"โหลดโมเดลแล้ว: {MODEL_ID}")
print(f"dtype จริงของ parameter: {next(model.parameters()).dtype}")
print(f"จำนวนพารามิเตอร์: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

# รันเบนช์มาร์กกลาง -- สัญญาการวัดผลถูกตรึงไว้ใน kobeval แล้ว
# (greedy, max_new_tokens=256, enable_thinking=False, seed=42)
baseline = evaluate(
    model,
    tokenizer,
    slices=["TH-KNOW", "TH-MATH", "TH-INSTR", "TH-SAFE"],
    seed=42,
    model_name=f"{MODEL_ID} (baseline)",
    out_path="results_baseline.json",
)

for name, s in baseline["slices"].items():
    print(
        f"{name:9s} acc={s['accuracy']:.3f} "
        f"[95% CI {s['ci_low']:.3f}-{s['ci_high']:.3f}]  "
        f"th_ratio={s['th_ratio']:.2f}  len={s['mean_output_len']:.0f}"
    )

print(f"\nรวมทุก slice: acc={baseline['overall']['accuracy']:.3f}  "
      f"th_ratio={baseline['overall']['th_ratio']:.2f}")


---

## 3. ทฤษฎีเท่าที่จำเป็น

_(เนื้อหาของตอนนี้)_


In [ ]:
# ตอนที่ 0 -- ส่วนที่ 3


---

## 4. เตรียมข้อมูล

_(เนื้อหาของตอนนี้)_


In [ ]:
# ตอนที่ 0 -- ส่วนที่ 4


---

## 5. ลงมือทำ (implementation)

_(เนื้อหาของตอนนี้)_


In [ ]:
# ตอนที่ 0 -- ส่วนที่ 5


---

## 6. รันจริง

_(เนื้อหาของตอนนี้)_


In [ ]:
# ตอนที่ 0 -- ส่วนที่ 6


---

## 7. วัดผลซ้ำด้วยชุดวัดเดิม

_(เนื้อหาของตอนนี้)_


In [ ]:
# ตอนที่ 0 -- ส่วนที่ 7


---

## 8. ตารางผลลัพธ์

ตารางมาตรฐานที่ใช้เหมือนกันทุกตอน สร้างจาก `compare()` ซึ่งอ่านค่าจริงจาก
`results.json` ไม่ใช่ตัวเลขที่พิมพ์เอง

แถบ error bar ในกราฟคือ **Wilson 95% confidence interval** ไม่ใช่ normal approximation
เหตุผลสำคัญ: แต่ละ slice มีแค่ 30 ข้อ ความต่างระหว่าง 40% กับ 47% ดูเหมือนดีขึ้น
จนกว่าจะวาดช่วงความเชื่อมั่นแล้วพบว่ามันซ้อนทับกันเกือบทั้งหมด


In [ ]:
# เปรียบเทียบ baseline กับผลหลังปรับปรุง
# (ในตอนที่เป็น template นี้ เราใช้ baseline ซ้ำสองครั้งเพื่อสาธิตรูปแบบตาราง
#  ในตอนจริงให้แทน `after` ด้วยผลของโมเดลหลังเทรน)
after = baseline  # TODO: แทนที่ด้วย evaluate(...) ของโมเดลหลังปรับปรุง

table = compare(baseline, after, out_path="results.json")

plot_before_after(
    baseline,
    after,
    slices=["TH-KNOW", "TH-MATH", "TH-INSTR"],
    title=f"ตอนที่ 0: ก่อน vs หลัง",
    out_path="before_after.png",
    results_json="results.json",
)


---

## 9. วิเคราะห์: อะไรได้ผล อะไรไม่ได้ผล

_(อภิปรายผล: ตัวเลขไหนขยับจริง ตัวเลขไหนขยับแค่ในสายตา)_

ข้อควรระวังในการอ่านผล:

- ดูค่า p จาก McNemar ที่ `compare()` พิมพ์ออกมาด้วย ไม่ใช่ดูแค่ accuracy
  accuracy ที่ขยับจาก 12/30 เป็น 14/30 อาจซ่อนการที่โมเดล "แก้ถูก 8 ข้อ
  แต่ทำพังเพิ่ม 6 ข้อ" ซึ่งไม่ใช่การพัฒนา
- ถ้า `th_ratio` ตก แปลว่าโมเดลเริ่มตอบเป็นภาษาอื่นมากขึ้น แม้ accuracy จะขึ้นก็ตาม

---

## 10. สรุป และตอนต่อไป

_(สรุปสิ่งที่ได้เรียนรู้ และเกริ่นตอนต่อไป)_


---

## ข้อจำกัดของการทดลองนี้

เขียนไว้ตรง ๆ เพื่อไม่ให้ตัวเลขข้างบนถูกอ่านเกินกว่าที่มันบอกได้จริง

1. **ขนาดชุดทดสอบเล็กมาก** — KobEval-TH มี slice ละ 30 ข้อเท่านั้น
   ช่วงความเชื่อมั่นจึงกว้าง ความต่างระดับ 1-2 ข้อ **ไม่ใช่** ความต่างที่มีนัยสำคัญ
   ตัวเลขในซีรีส์นี้ใช้เพื่อ "สอนวิธีวัด" ไม่ใช่เพื่อประกาศว่าโมเดลไหนดีที่สุด

2. **โมเดลเล็ก** — Qwen3-0.6B มีพารามิเตอร์เพียง 0.6B ข้อสรุปหลายอย่างที่ได้จาก
   โมเดลขนาดนี้ อาจไม่เป็นจริงกับโมเดลขนาด 7B ขึ้นไป

3. **รันครั้งเดียว ไม่ได้ทำ multiple seeds** — เราใช้ greedy decoding และ seed=42
   ตายตัวเพื่อให้ทำซ้ำได้ แต่ไม่ได้รายงานความแปรปรวนจากการเทรนหลาย seed
   ซึ่งในงานวิจัยจริงจำเป็นต้องทำ

4. **การให้คะแนนเป็นแบบอัตโนมัติทั้งหมด** — TH-KNOW ใช้ exact match,
   TH-INSTR ใช้ rubric ตายตัว, TH-SAFE ใช้การตรวจจับการปฏิเสธด้วย keyword
   วิธีพวกนี้เร็วและทำซ้ำได้ แต่หยาบกว่าการให้มนุษย์ตรวจ และมีทั้ง
   false positive และ false negative แน่นอน

5. **ฮาร์ดแวร์จำกัด** — ทุกอย่างถูกบีบให้รันได้บน T4 ฟรี ซึ่งแปลว่าต้องใช้
   fp16 แทน bf16, ใช้ sdpa แทน FlashAttention-2, batch size เล็ก และ
   sequence length สั้น ผลลัพธ์บนฮาร์ดแวร์ที่ใหญ่กว่าอาจต่างออกไป

6. **ยังไม่ได้ตรวจการปนเปื้อนของข้อมูล (contamination)** — ข้อสอบ TH-MATH
   เขียนขึ้นใหม่เองทั้งหมดเพื่อลดความเสี่ยงนี้ แต่ TH-KNOW เป็นความรู้ทั่วไป
   ที่อาจอยู่ในข้อมูลเทรนของโมเดลอยู่แล้ว

---

*ซีรีส์นี้เผยแพร่ภายใต้สัญญาอนุญาต Apache-2.0 — [kobkrit.com](https://kobkrit.com)*
